MovieLens Two Tower Model
- Simple Demo using in-batch negative sampling
    - Implicit Negative Sampling, no Explicit Negative Sampling for simplicity
    - aka we don't actually look at the rating value, but rather just whether or not the user watched the movie

Use predownloaded movielens_ratings dataset from movielens_download.py and load it into the notebook

In [1]:
import pandas as pd

column_names = ["user_id", "movie_id", "rating", "timestamp"]
ratings = pd.read_csv("movielens_ratings.csv", sep="\t", names=column_names)

ratings.head()

,user_id,movie_id,rating,timestamp
0,user_id,movie_id,rating,timestamp
1,196,242,3,881250949
2,186,302,3,891717742
3,22,377,1,878887116
4,244,51,2,880606923


Clean and Encode user and movie IDs into contiguous indicies

In [2]:
from sklearn.preprocessing import LabelEncoder

interactions = ratings[["user_id", "movie_id"]].copy()

user_encoder = LabelEncoder()
movie_encoder = LabelEncoder()

interactions["user"] = user_encoder.fit_transform(interactions["user_id"])
interactions["movie"] = movie_encoder.fit_transform(interactions["movie_id"])

num_users = interactions["user"].nunique()
num_movies = interactions["movie"].nunique()

print(f"# Users: {num_users}, # Movies: {num_movies}")
print(interactions.head())


# Users: 944, # Movies: 1683
   user_id  movie_id  user  movie
0  user_id  movie_id   943   1682
1      196       242   107    842
2      186       302    96    909
3       22       377   134    991
4      244        51   161   1139


Split train and test, standard 80/20

In [3]:
from sklearn.model_selection import train_test_split

# 80% train, 20% test
train_df, test_df = train_test_split(
    interactions[["user", "movie"]],
    test_size=0.2,
    random_state=42
)

print(f"Train size: {len(train_df)}, Test size: {len(test_df)}")


Train size: 80000, Test size: 20001


Build Towers
1. User Tower - User ID to embedding vector
2. Movie Tower - Movie ID to embedding vector
3. Final Similarity - Dot product of user and movie vector (should probably use something more sophisticated for the real deal)
4. Starting with initial embedding of 32, scale up for slower, more accurate, but more likely to overfit

In [4]:
import torch
import torch.nn as nn

class TwoTowerModel(nn.Module):
    # Scale embedding dimensions as needed here
    def __init__(self, num_users, num_movies, embedding_dim=32):
        super().__init__()
        self.user_embedding = nn.Embedding(num_users, embedding_dim)
        self.movie_embedding = nn.Embedding(num_movies, embedding_dim)

    def forward(self, user_ids, movie_ids):
        # Embed users and movies
        user_embeds = self.user_embedding(user_ids)
        movie_embeds = self.movie_embedding(movie_ids)

        # Compute dot product similarity
        return (user_embeds * movie_embeds).sum(dim=1)

    def get_user_embedding(self, user_ids):
        return self.user_embedding(user_ids)

    def get_movie_embedding(self, movie_ids):
        return self.movie_embedding(movie_ids)

Dataset/Loader Setup
1. A custom Dataset with (user_id,movie_id) pairs
2. DataLoader for batching/shuffling

In [6]:
from torch.utils.data import Dataset, DataLoader

class InteractionDataset(Dataset):
    def __init__(self, dataframe):
        # Long is used because nn.Embedding requires integer indices
        self.users = torch.tensor(dataframe["user"].values, dtype=torch.long)
        self.movies = torch.tensor(dataframe["movie"].values, dtype=torch.long)

    def __len__(self):
        return len(self.users)

    def __getitem__(self, idx):
        return self.users[idx], self.movies[idx]

# Create dataset and dataloader
train_dataset = InteractionDataset(train_df)
# Shuffle for training
# Batch of 256 to match the TF tutorial
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)


Train the model
- embed all movies and users in batch
- compute similarity matrix
- CrossEntropyLoss with diagonal as the correct movie
    - only implicit negatives
    - requires the correct movie at index i for user i

In [7]:
import torch
import torch.nn.functional as F
from torch import optim

# Set device
# For now this will just be cpu, but just in case
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Initialize model
embedding_dim = 32
model = TwoTowerModel(num_users=num_users, num_movies=num_movies, embedding_dim=embedding_dim).to(device)

# Optimizer
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Loss function: CrossEntropyLoss assumes logits + target index
loss_fn = nn.CrossEntropyLoss()

# Training loop
# Set epochs as needed
num_epochs = 5
for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    for user_batch, movie_batch in train_loader:
        user_batch = user_batch.to(device)
        movie_batch = movie_batch.to(device)

        # Get embeddings
        user_embeds = model.get_user_embedding(user_batch)     # [B, D]
        movie_embeds = model.get_movie_embedding(movie_batch)  # [B, D]

        # Compute similarity matrix: [B, B] = user_embeds @ movie_embeds.T
        logits = user_embeds @ movie_embeds.T

        # Targets are indices along the diagonal
        labels = torch.arange(len(user_batch)).to(device)

        # Compute loss
        loss = loss_fn(logits, labels)

        # Backprop
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{num_epochs} - Loss: {avg_loss:.4f}")


Epoch 1/5 - Loss: 15.1542
Epoch 2/5 - Loss: 13.3243
Epoch 3/5 - Loss: 11.9491
Epoch 4/5 - Loss: 10.8584
Epoch 5/5 - Loss: 9.9621


Evaluation
1. Generate embeddings for all users/movies
2. Compute similarity (dot)
3. Retrieve top K movie recommendations
4. Check if the user has watched movies that are recommended (recall@K)

In [10]:
from torch.utils.data import DataLoader
from tqdm import tqdm #progress bar

# Switch model to eval mode
model.eval()

# Generate full user and movie embeddings
with torch.no_grad():
    all_user_ids = torch.arange(num_users).to(device)
    all_movie_ids = torch.arange(num_movies).to(device)

    user_embeddings = model.get_user_embedding(all_user_ids)  # [num_users, dim]
    movie_embeddings = model.get_movie_embedding(all_movie_ids)  # [num_movies, dim]

# Compute full similarity matrix [num_users, num_movies]
similarity = user_embeddings @ movie_embeddings.T


Metric Eval
- This Recall@K is calculated as, for each user, whether at least one of their held-out (rated and in test) movies is in their top 10 recommendations
- Could later explore different recalls, MRR, MDCG, Precision@K etc
- Scalability with batch-wise evaluation?

In [11]:
K = 10
correct = 0
total = 0

# Create lookup: user -> list of held-out test movies
test_user_to_movie = test_df.groupby("user")["movie"].apply(list)

# Loop through all test users
for user_id, true_movies in test_user_to_movie.items():
    scores = similarity[user_id]  # [num_movies]
    top_k = torch.topk(scores, K).indices.tolist()  # Top-K movie indices

    # If any true movie is in the top-K predictions → hit
    if any(m in top_k for m in true_movies):
        correct += 1
    total += 1

recall_at_k = correct / total
print(f"Recall@{K}: {recall_at_k:.4f}")


Recall@10: 0.0276


Export results

In [12]:
# Number of recommendations
K = 10

# Get top-K movie indices for each user
topk_recs = {}

for user_id in range(num_users):
    scores = similarity[user_id]  # [num_movies]
    top_k = torch.topk(scores, K).indices.tolist()  # Movie indices
    topk_recs[user_id] = top_k


In [13]:
# Decode integer IDs back to original movie/user IDs
original_user_ids = user_encoder.inverse_transform(list(topk_recs.keys()))
decoded_recs = []

for internal_user_id, rec_movie_ids in topk_recs.items():
    user_id = original_user_ids[internal_user_id]
    for rank, internal_movie_id in enumerate(rec_movie_ids):
        movie_id = movie_encoder.inverse_transform([internal_movie_id])[0]
        decoded_recs.append({
            "user_id": user_id,
            "rank": rank + 1,
            "movie_id": movie_id
        })

# Create DataFrame
recommendation_df = pd.DataFrame(decoded_recs)

# Save to CSV
recommendation_df.to_csv("topk_recommendations.csv", index=False)
print("Saved recommendations to topk_recommendations.csv")


Saved recommendations to topk_recommendations.csv
